In [2]:
from langchain.chat_models import init_chat_model
#1.基于内存的持久化器
# 拥有记忆的智能体
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver

load_dotenv()
model=init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek"
)
response=model.invoke("你好")
print(response)

content='你好！很高兴见到你，有什么我可以帮你的吗？' additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer in Chinese likely. Need be careful: user says "你好" (Hello). We should respond friendly. Maybe introduce assistant. Need no mention constraints? Final only.'} response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 84, 'total_tokens': 133, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 36, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 84}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'e4db9e97-a738-4c35-97b6-b1cc1f5590af', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a07eb2-d3ce-7ae0-9ba6-b34a8a17d9b9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 84, 'output_tokens': 4

In [7]:
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
checkpointer=InMemorySaver()
agent=create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer#让agent或得了存储的能力
)
config={
    "configurable":{
        "thread_id":"1"
    }
}
response1=agent.invoke({
    "messages":[HumanMessage("我叫张三")]
},config=config)
print(f"agent:{response1['messages'][-1].content}")
response2=agent.invoke({
    "messages":[HumanMessage("我叫什么名字")]
},config=config)
print(f"agent:{response2['messages'][-1].content}")

agent:你好，张三！很高兴认识你。有什么我可以帮你的吗？
agent:你叫张三呀，刚才你告诉我的。 😊


In [8]:
response3=agent.invoke({
    "messages":[HumanMessage("我刚才问了什么问题")]
},config=config)
print(f"agent:{response3['messages'][-1].content}")

agent:你刚才问的是：“我叫什么名字？” 😄

再往前一点，你还告诉过我你叫张三呢。


In [ ]:
#不同的thead_id不共享内存

In [12]:
%pip install langgraph-checkpoint-sqlite

  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl (56 kB)
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.1
    Uninstalling langgraph-checkpoint-4.0.1:
      Successfully uninstalled langgraph-checkpoint-4.0.1
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver

# 数据库保存位置，文件夹不存在时自动创建
db_path = Path("D:/agent_memory/chat_memory.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

# 同一个 thread_id 对应同一个会话
config = {"configurable": {"thread_id": "1"}}

# 第二次测试时，把这里改成“我叫什么名字？”
question = "我叫什么名字"

with SqliteSaver.from_conn_string(str(db_path)) as checkpointer:
    agent = create_agent(
        model=model,  # 使用你已经配置好的模型
        tools=[],
        checkpointer=checkpointer,
    )

    response = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config,
    )

    print(f"agent:{response['messages'][-1].content}")

agent:你刚才告诉我你叫张三，所以我记得你叫张三。


In [13]:
#基于外部存储介质的持久化
from langgraph.checkpoint.postgres import PostgresSaver
from getpass import getpass
from psycopg.conninfo import make_conninfo
from langgraph.checkpoint.postgres import PostgresSaver

DB_URL = make_conninfo(
    host="127.0.0.1",
    port=15432,
    user="langchain_user",
    password=getpass("请输入 langchain_user 的数据库密码："),
    dbname="langchain_db",
    sslmode="disable",
    gssencmode="disable",
    connect_timeout=10,
)

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()
    print("连接成功，记忆存储表已初始化")
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
     checkpointer.setup()
     agent = create_agent(
         model, checkpointer=checkpointer
     )
     config = {"configurable": {"thread_id": "1"}}
     response1=agent.invoke({"messages":[HumanMessage("你好我叫张三！")]},config=config)
     for msg in response1["messages"]:
         msg.pretty_print()
     response2=agent.invoke({"messages":[HumanMessage("我是谁！")]},config=config)
     for msg in response1["messages"]:
         msg.pretty_print() 

连接成功，记忆存储表已初始化
================================ Human Message =================================

你好我叫张三！
================================== Ai Message ==================================

你好，张三！很高兴认识你。有什么我可以帮你的吗？ 😊
================================ Human Message =================================

你好我叫张三！
================================== Ai Message ==================================

你好，张三！很高兴认识你。有什么我可以帮你的吗？ 😊


基于内存的存储短期记忆不会在第二次启动时记忆
基于postgresql可以在重新启动时记忆